In [ ]:
from pathlib import Path

from segmentiq.data import get_data

In [ ]:
root = Path.cwd().parents[0].resolve()
raw_path = root / Path("data/raw/Online Retail.xlsx")
cache_path = root / Path("data/cache/retail.parquet")

In [ ]:
df = get_data(raw_path, cache_path)

.xlsx = 30 seconds / parquet = 0.133 seconds

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

We can see few things with this dataset. Firstly ,CustomerID is float we need to transform it into int.Secondly this dataset has missing values even tough data dictionary says opposite. In addition to missing values dataset has duplicates. And Finally from describe we can see negative values in unitprice and quantity columns. Before deletion in next pr i want to investigate this problems.

In [ ]:
missing_customer = df[df["CustomerID"].isna()]
missing_customer.head()

In [ ]:
missing_customer.describe()

In [ ]:
missing_customer["StockCode"].value_counts()

We can see that data without customerID is similar to the default data. Both of they have POST stock code for dotcom postage so they do not need customerId and i think they do not need to participate in our RFM analysis. also statistics are similar so i think there are missing values because of the guest purchases. However, in order to perform RFM clustering we need Customer ID to group by. Thus, i will drop missing rows for this column.

Also what i found is strange stock codes as S M POST DOT m Amazon FEE bank charge etc ... I will drop them cause they dont help with analysis and will only harm it. This codes refer to shipping prices and other fees not real orders.

In [ ]:
customer = df[~df["CustomerID"].isna()]
customer.isna().sum()

In [ ]:
customer.duplicated().sum()

Also by deleting all the rows without customerID strange rows with no description was also deleted so now we have clean dataset i will use to perform further EDA.However, duplicates remained in our dataset.To start with, I want to finish what i started with stockcodes for business operations that i will drop.

In [ ]:
customer["StockCode"].value_counts().sort_index(ascending=False)

In [ ]:
CODES = ["POST", "PADS", "M", "DOT", "D", "CRUK", "C2", "BANK CHARGES"]
print(customer[customer["StockCode"] == "AMAZONFEE"].shape)

In [ ]:
df.loc[df["StockCode"].isin(CODES), ["StockCode", "Description"]].drop_duplicates()

CODES = ["POST","M","DOT","D","CRUK","C2","BANK CHARGES"] In clean dataset there is no AMAZONFEE but other codes exists. Also pads is a real product that they sell so i wont drop it. Other codes are not products so i will drop them.

Now i want to look at the duplicates

In [ ]:
dup = customer[customer.duplicated(keep=False)]
dup.sort_values(["InvoiceNo", "StockCode", "Quantity"]).head(20)

Okay we can see that all duplicates are just same entries of one raw. Its safe to drop them in next PR.

Now let's look at the negative values

In [ ]:
negative_quantity = customer[customer["Quantity"] <= 0]
negative_quantity

In data dictionary we can see that invoice which start with "C" - means cancellation. I want to see if all canceled invoices have negative quantity

In [ ]:
print(f"number of negative quantity rows:{negative_quantity.shape[0]}")
print(
    f"number of negative quantity rows where invoice was canceled: {negative_quantity[negative_quantity['InvoiceNo'].str[0] == 'C'].shape[0]}"
)

In [ ]:
customer[(customer["Quantity"] <= 0) & (customer["UnitPrice"] <= 0)].shape

They are the same rows so all negative quantity rows are just cancellation we can drop them but it will be a mistake cause canceled order will give false monetary value for many customers.I will subscribe them in order to not face this problem. Also there is no rows with both negative unit price and quantity.

Last thing to check is negative unit price

In [ ]:
negative_price = customer[customer["UnitPrice"] < 0]
print(f"number of rows where unitprice is less than 0: {negative_price.shape[0]}")
print(
    f"number of rows where unitprice = 0: {customer[customer['UnitPrice'] == 0].shape[0]}"
)

We can see that after deletion of missing values in CustomerID there are no more observations where price < 0. Now we only have rows with price = zero that strange

In [ ]:
customer[customer["UnitPrice"] == 0]

Okay i think that price of zero is possible and is caused by some sort of promo actions or gifts and replacement of the product so i will keep them.

I think with strange unreal observations we have ended so now i will move on to the distributions of the price and quantity in order to find outliers in useful columns.